# 00 · Run all (end-to-end, single Kaggle session)

Data prep → QLoRA fine-tune → evaluation → GGUF export, all in **one session** so `/kaggle/working` persists across every step.

**Settings:** GPU **T4** on, **Internet** on. Runtime ~45–75 min.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Niikhi/peft-function-calling.git"
REPO_DIR = "peft-function-calling"
if not os.path.exists("src") and not os.path.exists(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(os.path.join(REPO_DIR, "src")):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())


In [ ]:
!pip -q install unsloth

### Log in to Hugging Face (for the gated xLAM dataset)
Accept the dataset terms first; skip to auto-use the ungated mirror.

In [ ]:
import os
os.environ["HF_HUB_REPO_ID"] = "Nikrobber/qwen2.5-3b-tool-calling"
from huggingface_hub import notebook_login
notebook_login()


## 1 · Prepare data

In [ ]:
from src.config import load_config
from src.data import load_records, split_records, save_jsonl

cfg = load_config()
splits = split_records(cfg, load_records(cfg))
for name, recs in splits.items():
    save_jsonl(recs, cfg.file_in("data_dir", f"{name}.jsonl"))
print({k: len(v) for k, v in splits.items()})

## 2 · Fine-tune

In [ ]:
from src.data import build_text_dataset, load_jsonl
from src.model import load_model_and_tokenizer, add_lora
from src.train import build_trainer, save_log_history, save_adapter, train_resumable

model, tokenizer = load_model_and_tokenizer(cfg)
model = add_lora(cfg, model)

train_ds = build_text_dataset(load_jsonl(cfg.file_in("data_dir", "train.jsonl")), tokenizer)
eval_ds  = build_text_dataset(load_jsonl(cfg.file_in("data_dir", "val.jsonl")), tokenizer)

trainer = build_trainer(cfg, model, tokenizer, train_ds, eval_ds)
train_resumable(cfg, trainer)
log_path = save_log_history(cfg, trainer)
save_adapter(cfg, model, tokenizer)


In [ ]:
from src.visualize import plot_loss_curves, plot_lr_schedule
from IPython.display import Image
plot_lr_schedule(log_path, cfg.file_in("figures_dir", "lr_schedule.png"))
Image(plot_loss_curves(log_path, cfg.file_in("figures_dir", "loss_curves.png")))

## 3 · Evaluate (base vs fine-tuned)
Frees the training model first to reclaim VRAM.

In [ ]:
import gc, torch
from src.evaluate import make_hf_generate_fn, run_eval
from src.model import load_model_and_tokenizer, for_inference
from huggingface_hub import snapshot_download

del model, trainer; gc.collect(); torch.cuda.empty_cache()
test_recs = load_jsonl(cfg.file_in("data_dir", "test.jsonl"))

adapter_dir = str(cfg.path("adapter_dir"))

def eval_one(weights, label):
    cfg.model["base_id"] = weights
    m, tok = load_model_and_tokenizer(cfg); for_inference(m)
    res = run_eval(cfg, test_recs, tok, make_hf_generate_fn(cfg, m, tok), label)
    del m; gc.collect(); torch.cuda.empty_cache()
    return res

results = {
    "base":      eval_one("unsloth/Qwen2.5-3B-Instruct-bnb-4bit", "base"),
    "finetuned": eval_one(adapter_dir, "finetuned"),
}


In [ ]:
from src.visualize import plot_metric_comparison, plot_error_breakdown
plot_error_breakdown(results, cfg.file_in("figures_dir", "error_breakdown.png"))
Image(plot_metric_comparison(results, cfg.file_in("figures_dir", "metric_comparison.png")))

In [ ]:
import pandas as pd
pd.DataFrame(results).T[["exact_match","name_accuracy","argument_f1","hallucination_rate","count_accuracy"]]

## 4 · Export GGUF quantizations

In [ ]:
import threading
from src.gguf_convert import merge_adapter_unsloth, run_full_conversion
import json

def build_llamacpp():
    if not os.path.exists("/kaggle/working/llama.cpp/build/bin/llama-quantize"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ggml-org/llama.cpp",
                        "/kaggle/working/llama.cpp"], check=True)
        subprocess.run(["cmake", "-B", "/kaggle/working/llama.cpp/build",
                        "-S", "/kaggle/working/llama.cpp", "-DGGML_CUDA=OFF"], check=True)
        subprocess.run(["cmake", "--build", "/kaggle/working/llama.cpp/build",
                        "--config", "Release", "-j"], check=True)

t = threading.Thread(target=build_llamacpp)
t.start()

cfg.model["base_id"] = adapter_dir
gmodel, gtok = load_model_and_tokenizer(cfg)
merged_dir = merge_adapter_unsloth(cfg, gmodel, gtok)

t.join()

from importlib import reload
import src.gguf_convert as gc_mod; reload(gc_mod)
from src.gguf_convert import run_full_conversion

rows = run_full_conversion(cfg, gtok, merged_dir, workdir=str(cfg.root))
ft_exact = results["finetuned"]["exact_match"]
for r in rows: r.setdefault("exact_match", ft_exact)
with open(cfg.file_in("metrics_dir", "gguf_manifest.json"), "w") as f:
    json.dump(rows, f, indent=2)

from src.visualize import plot_quant_quality_size
from IPython.display import Image
Image(plot_quant_quality_size(rows, cfg.file_in("figures_dir", "quant_quality_size.png")))


## 5 · Save outputs
Kaggle keeps `/kaggle/working` only if you **Save Version (Commit)**. The GGUF files + adapter + figures live under `outputs/` and `results/` there — download them from the committed version's **Output** tab.

In [ ]:
import os
for root in ("outputs", "results"):
    for dp, _, fns in os.walk(cfg.root / root):
        for fn in fns:
            print(os.path.join(dp, fn))